# 29 (MLA) — Estimator Bridge: Trees, KNN & Custom Models

**ML Analyst perspective.** Beyond linear models, IrisPark now trains **sklearn ensembles** server-side via Embedded Python (algorithm identity guaranteed — RandomForest trains RandomForest), and can plug **custom `IRISModel` classes** into the IntegratedML AutoML provider. CatBoost and TensorFlow plug in the same way (both expose the sklearn `BaseEstimator` contract).

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Data

300-row synthetic classification set with a clear signal.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
X = rng.normal(size=(300, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(float)
df = session.createDataFrame(pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "label": y}))
df.show(3)

## 2. Random Forest (EPython/sklearn backend)

`fit()` sends training data to Embedded Python, which fits sklearn and persists the model with joblib on IRIS; `transform()` loads and predicts. **Algorithm identity is guaranteed.**

In [ ]:
from irispark.ml.ensemble import RandomForestClassifier

rf = RandomForestClassifier(featuresCol=["x1", "x2"], labelCol="label", n_estimators=50)
model = rf.fit(df)
print("backend:", model.backend)
pred = model.transform(df)
pred.select("x1", "x2", "label", "prediction").show(5)

## 3. K-Nearest Neighbors

Same bridge, different sklearn estimator.

In [ ]:
from irispark.ml.ensemble import KNeighborsClassifier

knn = KNeighborsClassifier(featuresCol=["x1", "x2"], labelCol="label", n_neighbors=5)
km = knn.fit(df)
print("backend:", km.backend)
km.transform(df).select("label", "prediction").show(5)

## 4. Custom Model via IntegratedML AutoML

Write an `IRISModel` wrapper (KNN, sklearn contract) into `AutoML/Classifiers`, then `CREATE MODEL`/`TRAIN MODEL`/`PREDICT` by name. AutoML selects among candidates; needs enough training rows and polls a fresh connection (LESSONS remedies).

**Note**: the provider requires the `intersystems-iris-automl` Python package in `mgr/python` (provisioned by our docker entrypoint). `MaxTime` is measured in **minutes** and only applies with `TrainMode: "TIME"` — both are set below, so a cold instance is bounded server-side; if training still can't complete, the cell skips gracefully instead of hanging.

In [ ]:
from irispark.ml.custom import CustomModelClassifier

try:
    cm = CustomModelClassifier(featuresCol=["x1", "x2"], labelCol="label", n_neighbors=5, maxTime=2, pollTimeout=90).fit(df)
    print("custom model:", cm.model_name)
    cm.transform(df).select("label", "prediction").show(5)
    session.sql("DROP MODEL " + cm.model_name)
except TimeoutError:
    print("SKIP: IntegratedML custom-model training did not complete in this environment.")
    print("The EPython/sklearn bridge (sections 2-3) covers the same use case reliably.")

## 5. Drop-ins: CatBoost & TensorFlow

Both expose the sklearn `BaseEstimator` contract, so they plug into the same `IRISModel` bridge. Install into `mgr/python`, write an `IRISModel` wrapper, reference by name. See `docs/` for the full contract (`name`, `model`, `classes_` in `fit()`).

In [ ]:
print("CatBoost/TensorFlow: same IRISModel contract - install + wrap + reference by name")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")